# Import and explore the data

In [51]:
import pandas as pd
from sklearn.feature_selection import f_classif
from sklearn.model_selection import train_test_split

raw_train = pd.read_csv('../data/train.csv', index_col="ID")
raw_clientes = pd.read_csv('../data/clientes.csv', index_col="customer_id")

raw_data = pd.merge(raw_train, raw_clientes, how='left', on='customer_id')


In [24]:
print(f"{raw_train.shape[0]:,} rows x {raw_train.shape[1]} feature columns, indexed by RecordID")

6,534 rows x 38 feature columns, indexed by RecordID


In [ ]:
raw_train

In [ ]:
print(raw_train.head())
print("\ninfo of the training data:")
print(raw_train.info())
print("\ndescriptive statistics of the training data:")
print(raw_train.describe())
print("\nna values in each column:n")
print(raw_train.isna().sum())

In [ ]:
print(raw_clientes.head())
print("\ninfo of the training data:")
print(raw_clientes.info())
print("\ndescriptive statistics of the training data:")
print(raw_clientes.describe())
print("\nna values in each column:n")
print(raw_clientes.isna().sum())

In [25]:
raw_train[["repeat_purchase_90d", "random_noise"]].corr()

,repeat_purchase_90d,random_noise
repeat_purchase_90d,1.000000,-0.011988
random_noise,-0.011988,1.000000


In [31]:
raw_train["random_noise"] = raw_train["random_noise"].fillna(raw_train["random_noise"].mean())

In [34]:
F, p = f_classif(raw_train[[ "random_noise" ]], raw_train["repeat_purchase_90d"])

print(F)
print(p)

[0.92936305]
[0.33506418]


In [20]:
raw_train.groupby("repeat_purchase_90d")["random_noise"].describe()

,count,mean,std,min,25%,50%,75%,max
repeat_purchase_90d,,,,,,,,
0,2848.0,-0.008609,0.996088,-3.529210,-0.642745,-0.043238,0.635513,4.151241
1,3621.0,-0.032783,1.004952,-3.263267,-0.715974,-0.047517,0.649046,3.435928


In [41]:
target = raw_train["repeat_purchase_90d"]


In [39]:
raw_train.drop(columns=["repeat_purchase_90d"], inplace=True)
raw_data.drop(columns=["repeat_purchase_90d"], inplace=True)

In [42]:
raw_data["repeat_purchase_90d"].value_counts(normalize=True)

repeat_purchase_90d
1    0.559229
0    0.440771
Name: proportion, dtype: float64

In [43]:
raw_data.isna().sum()

Total a Pagar                       65
Total Bruto                         65
Total Liquido                       65
Total Imp_                          65
Total Portes                        65
Total Desc_ Global                  65
Total Desc_ Linha                   65
Custo                               65
N_ Detalhes                         65
invoice_count_current_purchase      65
repeat_purchase_90d                  0
days_since_previous_purchase       861
customer_tenure_days                65
prior_purchase_count                65
prior_total_spend                   65
prior_mean_purchase_value          862
prior_std_purchase_value          1113
prior_min_purchase_value           668
prior_max_purchase_value           668
prior_mean_gap_days               1308
prior_std_gap_days                1478
purchase_count_30d                  65
spend_30d                           65
purchase_count_90d                  65
spend_90d                           65
purchase_count_180d      

In [48]:
raw_data.isna().sum()


To gain an initial understanding of the data structure, data types, and quality, we performed a exploratory analysis using typicall methods such as head(), info(), describe(), and null-value checking that we learned in class. Here´s what we discover: 

Training Dataset (train.csv) 

- **Dimensions**: Comprises 6,534 purchase occasions (Observations/rows) and 39 variables (Columns).  
- **Target Variable**: The target variable "repeat_purchase_90d" contains 0 missing values, as we can see in info(), which is good. 
- **Missing Values and Anomalies**: 
  - Financial columns (e.g., "Total a Pagar", "Custo") exhibit exactly **65 missing values** (6534 - 6469), indicating rows with incomplete data.
  - Historical temporal columns (such as "days_since_previous_purchase" and "prior_mean_gap_days") show a higher number of nulls.
    
Customer Dataset (clientes.csv) 

- **Dimensions**: Contains **1,143 unique customer profiles** mapped through "customer_id".
- This Dataset also has some missing values including a column with no data


# Clean and pre-process the data

In [52]:

# 1. Fazer o merge com a tabela de clientes (excluindo a coluna 100% vazia 'customer_type')
clientes_clean = raw_clientes.drop(columns=['customer_type'])
train_merged = raw_train.merge(clientes_clean, on='customer_id', how='left')

# 2. Tratar valores em falta nas colunas numéricas (preenchendo com 0)
numeric_cols = train_merged.select_dtypes(include=['float64', 'int64']).columns
train_merged[numeric_cols] = train_merged[numeric_cols].fillna(0)

# 3. Tratar valores em falta nas colunas de texto (preenchendo com 'Unknown')
categorical_cols = train_merged.select_dtypes(include=['object', 'str']).columns
train_merged[categorical_cols] = train_merged[categorical_cols].fillna('Unknown')

# 4. Remover colunas desnecessárias ou identificadores (como ID, ruído e datas em formato string)
cols_to_drop = ['ID', 'random_noise', 'purchase_date']
train_clean = train_merged.drop(columns=[col for col in cols_to_drop if col in train_merged.columns])

# 5. Separar a variável alvo (y) das variáveis preditivas (X) a partir da tabela limpa
X = train_clean.drop(columns=['repeat_purchase_90d'])
y = train_clean['repeat_purchase_90d']

# 6. Codificar as variáveis categóricas em formato numérico (One-Hot Encoding)
X = pd.get_dummies(X, drop_first=True)

# 7. Split em treino e validação (70% / 30%) com estratificação
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    train_size=0.7,      # 70% para treino, 30% para validação
    shuffle=True,        
    stratify=y,          # mantém o rácio da classe alvo em ambas as partes
    random_state=42      # semente fixa para reprodutibilidade
)

# 8. Reportar os tamanhos resultantes
print(f"X_train size: {len(X_train)}, X_val size: {len(X_val)}")

X_train size: 4573, X_val size: 1961


# Feature selection

# Build a simple model and assess performance